# Thai Sign Language - Train PoseToTextT5
Runs the latest staged code dataset against the portable mixed-all-train v6 dataset on Kaggle GPU and resumes from the best-state TSL-51 seed dataset when present.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

def find_dataset(slug: str) -> str:
    candidates = [
        f'/kaggle/input/{slug}',
        f'/kaggle/input/datasets/orbitorls/{slug}',
    ]
    for candidate in candidates:
        if os.path.isdir(candidate):
            return candidate
    return candidates[0]

CODE_DATASET = find_dataset('thai-sign-code')
DATA = find_dataset('thai-sign-mixed-all-v6-archived')
DATA_RUNTIME = DATA
MT5_LOCAL = find_dataset('thai-sign-mt5small')
PREV_CKPT = find_dataset('thai-sign-tsl51-seed')
USE_PREVIOUS_CHECKPOINT = True
CKPT_DIR = '/kaggle/working/pose_t5_mixed_all_v6'
CODE = '/tmp/thai_sign_code'
os.makedirs(CKPT_DIR, exist_ok=True)
shutil.rmtree(CODE, ignore_errors=True)
os.makedirs(CODE, exist_ok=True)

repo_bundle = os.path.join(CODE_DATASET, 'repo_bundle.zip')
if os.path.isfile(repo_bundle):
    with zipfile.ZipFile(repo_bundle) as archive:
        archive.extractall(CODE)
else:
    for name in ['config.py', 'requirements.txt', 'README.md']:
        src = os.path.join(CODE_DATASET, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(CODE, name))
    for name in ['src', 'scripts']:
        src_dir = os.path.join(CODE_DATASET, name)
        if os.path.isdir(src_dir):
            shutil.copytree(src_dir, os.path.join(CODE, name), dirs_exist_ok=True)
    for name in ['src.zip', 'scripts.zip']:
        archive_path = os.path.join(CODE_DATASET, name)
        if os.path.isfile(archive_path):
            with zipfile.ZipFile(archive_path) as archive:
                archive.extractall(CODE)

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'],
    capture_output=True,
    text=True,
    timeout=30,
)
if result.returncode != 0:
    raise RuntimeError('No GPU detected in this Kaggle session.')
gpu_name, gpu_cap, gpu_mem = [part.strip() for part in result.stdout.strip().split('\n')[0].split(',')]
allow_legacy_sm60 = os.environ.get('TSL_ALLOW_LEGACY_SM60', '0').strip().lower() in {'1', 'true', 'yes'}
use_cu126_torch = float(gpu_cap) < 7.0 and allow_legacy_sm60
if float(gpu_cap) < 7.0 and not allow_legacy_sm60:
    raise RuntimeError(f'Rejecting unsupported GPU {gpu_name} (sm_{gpu_cap.replace(".", "")}). Use T4/V100/A100, or set TSL_ALLOW_LEGACY_SM60=1 explicitly for a non-default legacy run.')
if use_cu126_torch:
    print(f'Explicit legacy sm_60 override enabled; installing CUDA 12.6 PyTorch wheel for {gpu_name}.')

for package in ['transformers>=4.40', 'sentencepiece>=0.2.0', 'sacrebleu>=2.4.0', 'portalocker>=2.0', 'pandas>=2.0']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if use_cu126_torch:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])

features_archive = os.path.join(DATA, 'features.zip')
if os.path.isfile(features_archive):
    data_runtime = Path('/kaggle/working/datasets') / Path(DATA).name
    shutil.rmtree(data_runtime, ignore_errors=True)
    data_runtime.mkdir(parents=True, exist_ok=True)
    for name in ['manifest.csv', 'manifest_quality.json']:
        src = os.path.join(DATA, name)
        if os.path.isfile(src):
            shutil.copy2(src, data_runtime / name)
    feature_root = data_runtime / 'features'
    feature_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(features_archive) as archive:
        archive.extractall(feature_root)
    DATA_RUNTIME = str(data_runtime)

if USE_PREVIOUS_CHECKPOINT and os.path.isdir(PREV_CKPT):
    for src in Path(PREV_CKPT).iterdir():
        if src.is_file():
            dst = Path(CKPT_DIR) / src.name
            if not dst.exists():
                shutil.copy2(src, dst)

print({'gpu_name': gpu_name, 'gpu_cap': gpu_cap, 'gpu_mem': gpu_mem, 'use_cu126_torch': use_cu126_torch, 'code_dataset': CODE_DATASET, 'code_runtime': CODE, 'data': DATA, 'data_runtime': DATA_RUNTIME, 'mt5': MT5_LOCAL, 'resume': PREV_CKPT if USE_PREVIOUS_CHECKPOINT else 'disabled'})

In [ ]:
import os
import shutil
from pathlib import Path
import pandas as pd
import torch

manifest_path = os.path.join(DATA_RUNTIME, 'manifest.csv')
if not os.path.isfile(manifest_path):
    raise FileNotFoundError(f'manifest.csv not found: {manifest_path}')

df = pd.read_csv(manifest_path)
if 'npy_path' not in df.columns:
    raise ValueError(f'npy_path column missing from {manifest_path}')
features_dir = os.path.join(DATA_RUNTIME, 'features')
sample_paths = [os.path.join(DATA_RUNTIME, str(value)) for value in df['npy_path'].head(min(8, len(df)))]
missing_paths = [path for path in sample_paths if not os.path.isfile(path)]
if missing_paths and os.path.isdir(features_dir):
    patched_manifest_dir = Path('/kaggle/working/datasets') / f'{Path(DATA).name}_manifestfix'
    shutil.rmtree(patched_manifest_dir, ignore_errors=True)
    patched_manifest_dir.mkdir(parents=True, exist_ok=True)
    for name in ['manifest_quality.json']:
        src = os.path.join(DATA, name)
        if os.path.isfile(src):
            shutil.copy2(src, patched_manifest_dir / name)
    df['npy_path'] = [
        str((Path(features_dir) / str(value)).resolve()) if os.path.isfile(os.path.join(features_dir, str(value))) else str(value)
        for value in df['npy_path']
    ]
    manifest_path = patched_manifest_dir / 'manifest.csv'
    df.to_csv(manifest_path, index=False)
    DATA_RUNTIME = str(patched_manifest_dir)
    sample_paths = [str(value) if os.path.isabs(str(value)) else os.path.join(DATA_RUNTIME, str(value)) for value in df['npy_path'].head(min(8, len(df)))]
    missing_paths = [path for path in sample_paths if not os.path.isfile(path)]
if missing_paths:
    raise FileNotFoundError({'dataset_root': DATA_RUNTIME, 'missing_feature_samples': missing_paths})
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'Rows: {len(df)}')
print(f'Feature sample check: {len(sample_paths) - len(missing_paths)}/{len(sample_paths)} present')
if 'source' in df.columns:
    print(df['source'].value_counts().to_dict())
print(df.head(3))

In [ ]:
import os, subprocess, sys, torch

base_model = MT5_LOCAL if os.path.isdir(MT5_LOCAL) else 'google/mt5-small'
gpu_mem_gb = 0.0
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
batch_size = 4
grad_accum = 4

train_script_candidates = [
    os.path.join(CODE, 'scripts', 'kaggle_train_pose_t5.py'),
    os.path.join(CODE, 'kaggle_train_pose_t5.py'),
]
train_script = next((path for path in train_script_candidates if os.path.isfile(path)), None)
if train_script is None:
    raise FileNotFoundError({'searched': train_script_candidates, 'code_contents': sorted(os.listdir(CODE))})

cmd = [
    sys.executable,
    train_script,
    '--data-roots', DATA_RUNTIME,
    '--out-dir', CKPT_DIR,
    '--base-model', base_model,
    '--epochs', '120',
    '--batch-size', str(batch_size),
    '--grad-accum', str(grad_accum),
    '--amp', 'auto',
    '--max-runtime-min', '690',
    '--eval-steps', '200',
    '--checkpoint-steps', '200',
    '--resume', 'best_state',
    '--lr', '5e-5',
    '--dropout', '0.4',
    '--weight-decay', '0.1',
    '--require-gpu', 'true',
    '--smoke-steps', '20',
    '--balance-sources', 'auto',
    '--focus-target-tokens', 'ฉัน,คุณ,แม่,พี่,วันนี้,พรุ่งนี้',
    '--focus-target-max-multiplier', '3.0',
    '--preload-train-features', 'false',
    '--required-sources', 'tsl51',
    '--manifest-quality-sources', 'tsl51',
    '--fail-on-manifest-quality', 'true',
    '--allow-noop-resume', 'false',
    '--split-policy', 'manifest',
    '--early-stopping-metric', 'val_chrf',
    '--early-stopping-patience', '6',
    '--reset-progress-history',
    '--evaluate-after-train', 'true',
    '--eval-data-roots', DATA_RUNTIME,
    '--eval-report-data-roots', 'data/mixed_all_train_v6',
    '--eval-sources', 'tsl51',
    '--eval-split-policy', 'manifest',
    '--eval-device', 'cpu',
    '--eval-val-subset-size', '50',
    '--min-val-chrf', '80',
    '--min-val-exact-match-pct', '80',
]
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([os.path.join(CODE, 'src'), CODE, env.get('PYTHONPATH', '')])
if os.path.isdir(MT5_LOCAL):
    env['TRANSFORMERS_OFFLINE'] = '1'
    env['HF_HUB_OFFLINE'] = '1'
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)

In [ ]:
import os, subprocess, sys

search_script_candidates = [
    os.path.join(CODE, 'scripts', 'search_pose_t5_decoding.py'),
    os.path.join(CODE, 'search_pose_t5_decoding.py'),
]
search_script = next((path for path in search_script_candidates if os.path.isfile(path)), None)
if search_script is None:
    raise FileNotFoundError({'searched': search_script_candidates, 'code_contents': sorted(os.listdir(CODE))})

search_cmd = [
    sys.executable,
    search_script,
    '--export-dir', CKPT_DIR,
    '--data-roots', DATA_RUNTIME,
    '--report-data-roots', 'data/mixed_all_train_v6',
    '--required-sources', 'tsl51',
    '--manifest-quality-sources', 'tsl51',
    '--eval-sources', 'tsl51',
    '--split-policy', 'manifest',
    '--val-subset-size', '25',
    '--device', 'cpu',
    '--max-trials', '32',
]
print('Running:', ' '.join(search_cmd))
subprocess.run(search_cmd, check=True, env=env)

In [ ]:
import os
files = sorted(os.listdir(CKPT_DIR)) if os.path.isdir(CKPT_DIR) else []
print(files)
print(f'Total files: {len(files)}')